In [ ]:
# Proyecto Analisis Multivariado

# Importo las librer´ıas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

class Regresion_LinealPesada() -> np.ndarray:

  def __init__(self,X):

      self.x = np.array(X) # Baja a la anotacion que hice abajo

  def kernel_Gass(self,x_hat,tau) -> np.ndarray:
      """
      Calcula los pesos gaussianos para un conjunto de valores x_values
      respecto a un punto de referencia x_hat.

      Params:
      - x_values: lista o array de valores (x^i)
      - x_hat: valor central alrededor del cual se calculan los pesos
      - tau: parámetro de suavizado (controla la flexibilidad)

      Returns:
      - np.array con los pesos correspondientes
      """
      distancia = self.X - x_hat #pusiste self.X en lugar de de self.x
      weights = np.exp(-(distancia**2) / (2 * tau**2))
      return weights

  def tricube_weight_function(self,X_hat) -> np.ndarray:
      """
      Función de peso tricube utilizada en regresion local.
      """
      abs_distance = np.abs(self.X - X_hat)
      weights = (1 - abs_distance**3) ** 3
      weights[abs_distance >= 1] = 0.0
      return weights


  def Matriz(self,x_hat,tau,tipo_kernel="Gauss") -> np.ndarray:
    """
    Funcion para armar la matriz
    """
    if tipo_kernel == "Gauss":
      kernel = self.kernel_Gass(x_hat,tau)
    elif tipo_kernel == "Tricube":
      kernel = self.tricube_weight_function(x_hat)
    else:
      ValueError("Error")
    return np.diag(kernel)






In [ ]:
# Testeo Aparte

class Regresion_LinealPesada:

    def __init__(self, X):
        self.x = np.array(X)

    def kernel_Gauss(self, x_hat, tau):

        distancia = self.x - x_hat

        weights = np.exp(-(distancia**2) / (2 * tau**2))

        return weights

    def tricube_weight_function(self, x_hat):

        abs_distance = np.abs(self.x - x_hat)

        weights = (1 - abs_distance**3) ** 3

        weights[abs_distance >= 1] = 0.0

        return weights

    def Matriz(self, x_hat, tau, tipo_kernel="Gauss"):

        if tipo_kernel == "Gauss":
            kernel = self.kernel_Gauss(x_hat, tau)

        elif tipo_kernel == "Tricube":
            kernel = self.tricube_weight_function(x_hat)

        else:
            raise ValueError("Kernel no válido")

        return np.diag(kernel)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn import datasets

class Regresion_LinealPesada():

    def __init__(self, X, y):
        self.X = np.array(X)
        self.y = np.array(y)

    def kernel_Gass(self, x_hat, tau) -> np.ndarray:
        """
        Calcula los pesos gaussianos basándose en la distancia euclidiana
        multivariada entre cada fila de self.X y el punto x_hat.
        """
        distancia = np.linalg.norm(self.X - x_hat, axis=1)
        weights = np.exp(-(distancia**2) / (2 * tau**2))
        return weights

    def tricube_weight_function(self, x_hat) -> np.ndarray:
        """
        Función de peso tricube utilizada en regresión local.
        """
        distancia = np.linalg.norm(self.X - x_hat, axis=1)

        # Normalizamos la distancia para que el máximo sea 1 (evita pesos negativos)
        max_dist = np.max(distancia) if np.max(distancia) > 0 else 1
        u = distancia / max_dist

        weights = (1 - u**3) ** 3
        weights[u >= 1] = 0.0
        return weights

    def Matriz(self, x_hat, tau, tipo_kernel="Gauss") -> np.ndarray:
        """
        Función para armar la matriz diagonal de pesos W.
        """
        if tipo_kernel == "Gauss":
            kernel = self.kernel_Gass(x_hat, tau)
        elif tipo_kernel == "Tricube":
            kernel = self.tricube_weight_function(x_hat)
        else:
            raise ValueError("Error: Kernel no reconocido") # Corregido: se usa 'raise'
        return np.diag(kernel)

    def predict(self, x_hat, tau=0.5, tipo_kernel="Gauss") -> float:
        """
        Estima el valor de y para un punto específico x_hat usando WLS local.
        """
        W = self.Matriz(x_hat, tau, tipo_kernel)

        try:
            parte_inversa = np.linalg.inv(self.X.T @ W @ self.X)
            beta = parte_inversa @ self.X.T @ W @ self.y

            return float(x_hat @ beta)
        except np.linalg.linalg.LinAlgError:
            return np.nan


iris = datasets.load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

X_df = df[['sepal_length', 'sepal_width', 'petal_length']]
y_df = df['petal_width']

X_with_const = sm.add_constant(X_df)

modelo = Regresion_LinealPesada(X_with_const, y_df)

punto_prueba = np.array(X_with_const.iloc[0]) # [constante, sepal_length, sepal_width, petal_length]
valor_real = y_df.iloc[0]

print("--- VERIFICACIÓN DE COMPONENTES ---")
pesos_gauss = modelo.kernel_Gass(punto_prueba, tau=0.7)
print(f"Forma de vector de pesos Gauss: {pesos_gauss.shape} (Debe ser 150,)")

W_matriz = modelo.Matriz(punto_prueba, tau=0.7, tipo_kernel="Tricube")
print(f"Forma de la matriz W: {W_matriz.shape} (Debe ser 150x150)")

print("\n--- RESULTADO DE LA PREDICCIÓN ---")
# Calcular la predicción local para ese punto específico
prediccion_gauss = modelo.predict(punto_prueba, tau=0.7, tipo_kernel="Gauss")
prediccion_tricube = modelo.predict(punto_prueba, tau=0.7, tipo_kernel="Tricube")

print(f"Valor Real (Petal Width): {valor_real}")
print(f"Predicción con Kernel Gaussiano: {prediccion_gauss:.4f}")
print(f"Predicción con Kernel Tricube: {prediccion_tricube:.4f}")

print("\n--- FRAGMENTO DE LA MATRIZ W (Primeras 5x5 posiciones) ---")
# Configuramos pandas/numpy para que muestre decimales limpios
np.set_printoptions(precision=4, suppress=True)

# Imprimir solo las primeras 5 filas y 5 columnas
print(W_matriz[:5, :5])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

class Regresion_LinealPesada():

    def __init__(self, X, y):
        self.X = np.array(X)
        self.y = np.array(y)

    def kernel_Gass(self, x_hat, tau) -> np.ndarray:
        """
        Calcula los pesos gaussianos basándose en la distancia euclidiana
        multivariada entre cada fila de self.X y el punto x_hat.
        """
        distancia = np.linalg.norm(self.X - x_hat, axis=1)
        weights = np.exp(-(distancia**2) / (2 * tau**2))
        return weights

    def tricube_weight_function(self, x_hat) -> np.ndarray:
        """
        Función de peso tricube utilizada en regresión local.
        """
        distancia = np.linalg.norm(self.X - x_hat, axis=1)

        # Normalizamos la distancia para que el máximo sea 1 (evita pesos negativos)
        max_dist = np.max(distancia) if np.max(distancia) > 0 else 1
        u = distancia / max_dist

        weights = (1 - u**3) ** 3
        weights[u >= 1] = 0.0
        return weights

    def Matriz(self, x_hat, tau, tipo_kernel="Gauss") -> np.ndarray:
        """
        Función para armar la matriz diagonal de pesos W.
        """
        if tipo_kernel == "Gauss":
            kernel = self.kernel_Gass(x_hat, tau)
        elif tipo_kernel == "Tricube":
            kernel = self.tricube_weight_function(x_hat)
        else:
            raise ValueError("Error: Kernel no reconocido")
        return np.diag(kernel)

    def predict(self, x_hat, tau=0.5, tipo_kernel="Gauss") -> float:
        """
        Estima el valor de y para un punto específico x_hat usando WLS local.
        """
        W = self.Matriz(x_hat, tau, tipo_kernel)

        try:
            parte_inversa = np.linalg.inv(self.X.T @ W @ self.X)
            beta = parte_inversa @ self.X.T @ W @ self.y
            return float(x_hat @ beta)
        except np.linalg.linalg.LinAlgError:
            return np.nan


# ========================================================
# 1. CARGAR DATASET FISH MARKET DESDE REPOSITORIO PÚBLICO
# ========================================================
url = "Fish.csv"
df = pd.read_csv(url)

# Eliminamos la columna categórica 'Species' para trabajar puramente con regresión numérica continua
df_numeric = df.drop(columns=['Species'])

# 2. Definir variables (X: Dimensiones físicas, y: Peso del pescado)
# Variables independientes: Length1, Length2, Length3, Height, Width
X_df = df_numeric.drop(columns=['Weight'])
y_df = df_numeric['Weight']

# En regresiones locales/pesadas, la diferencia de escalas altera drásticamente las distancias.
# Escalamos los datos X (Z-score) para que las variables tengan el mismo peso geométrico.
X_scaled = (X_df - X_df.mean()) / X_df.std()

# Añadir la constante (intersección)
X_with_const = sm.add_constant(X_scaled)

# 3. Inicializar el modelo con Fish Market
modelo = Regresion_LinealPesada(X_with_const, y_df)

# Definimos el primer pescado del dataset como nuestro punto de prueba
punto_prueba = np.array(X_with_const.iloc[0])
valor_real = y_df.iloc[0]

# Ajustamos un tau más alto (e.g., 2.5) debido a la dispersión tras el escalado geométrico
tau_valor = 2.5

print(f"--- VERIFICACIÓN DE COMPONENTES (FISH MARKET, N={len(df)}) ---")
pesos_gauss = modelo.kernel_Gass(punto_prueba, tau=tau_valor)
print(f"Forma de vector de pesos Gauss: {pesos_gauss.shape} (Debe ser {len(df)},)")

W_matriz = modelo.Matriz(punto_prueba, tau=tau_valor, tipo_kernel="Tricube")
print(f"Forma de la matriz W: {W_matriz.shape} (Debe ser {len(df)}x{len(df)})")

print("\n--- RESULTADO DE LA PREDICCIÓN (PESO EN GRAMOS) ---")
prediccion_gauss = modelo.predict(punto_prueba, tau=tau_valor, tipo_kernel="Gauss")
prediccion_tricube = modelo.predict(punto_prueba, tau=tau_valor, tipo_kernel="Tricube")

print(f"Valor Real (Weight): {valor_real} g")
print(f"Predicción con Kernel Gaussiano: {prediccion_gauss:.4f} g")
print(f"Predicción con Kernel Tricube: {prediccion_tricube:.4f} g")

print("\n--- FRAGMENTO DE LA MATRIZ W DE PECES (Primeras 5x5 posiciones) ---")
np.set_printoptions(precision=4, suppress=True)
print(W_matriz[:5, :5])


# ========================================================
# 4. GRÁFICO DE DISTRIBUCIÓN Y CORTE DE PESOS (PECES)
# ========================================================
# Calculamos la distancia euclidiana omitiendo la columna de la constante
distancias = np.linalg.norm(modelo.X[:, 1:] - punto_prueba[1:], axis=1)

pesos_gauss_grafico = modelo.kernel_Gass(punto_prueba, tau=tau_valor)
pesos_tricube_grafico = np.diag(modelo.Matriz(punto_prueba, tau=tau_valor, tipo_kernel="Tricube"))

indices_ordenados = np.argsort(distancias)
dist_ordenadas = distancias[indices_ordenados]
gauss_ordenado = pesos_gauss_grafico[indices_ordenados]
tricube_ordenado = pesos_tricube_grafico[indices_ordenados]

plt.figure(figsize=(10, 6))

# Pintamos cada pez mapeado en la gráfica
plt.scatter(distancias, pesos_gauss_grafico, color='teal', alpha=0.4, label='Peces (Pesos Gauss)')
plt.scatter(distancias, pesos_tricube_grafico, color='darkorange', alpha=0.4, label='Peces (Pesos Tricube)')

# Curvas de decaimiento
plt.plot(dist_ordenadas, gauss_ordenado, color='teal', linestyle='-', linewidth=2, label=f'Kernel Gaussiano (tau={tau_valor})')
plt.plot(dist_ordenadas, tricube_ordenado, color='darkorange', linestyle='--', linewidth=2, label='Kernel Tricube')

plt.axhline(0, color='black', linestyle=':', alpha=0.5)

plt.title('Dataset Fish Market: Comparativa de Pesos Locales\n(Respecto al Pescado de Prueba)', fontsize=12, fontweight='bold')
plt.xlabel('Distancia Geométrica al Pez Objetivo (Variables Físicas Escaladas)', fontsize=10)
plt.ylabel('Magnitud del Peso asignado en la Matriz W', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')

plt.show()